In [ ]:
#import librerie
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Librerie caricate correttamente")

Librerie caricate correttamente


In [ ]:
# Caricare il file
from google.colab import files
import io

uploaded = files.upload()  # si apre una finestra per caricare il file
filename = list(uploaded.keys())[0]
print(f"File caricato: {filename}")

Saving 2. Dati comunali 2014-2024.xlsx to 2. Dati comunali 2014-2024.xlsx
File caricato: 2. Dati comunali 2014-2024.xlsx


In [ ]:
import io
FILE_ANNUALE = io.BytesIO(uploaded[filename])
ANNI = list(range(2014, 2025))
print(f"File pronto: {filename}")
print(f"Anni da caricare: {ANNI}")

File pronto: 2. Dati comunali 2014-2024.xlsx
Anni da caricare: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [ ]:
#Caricamento degli anni
def carica_anno(filepath, anno):
    df = pd.read_excel(filepath, sheet_name=str(anno), header=None, skiprows=6)
    df.columns = [
        'cod_reg', 'regione', 'cod_prov', 'provincia', 'comune', 'cod_istat', 'flag',
        'arr_tot_res', 'arr_tot_nonres', 'arr_tot_tot',
        'arr_alb_res', 'arr_alb_nonres', 'arr_alb_tot',
        'arr_ext_res', 'arr_ext_nonres', 'arr_ext_tot',
        'col16',
        'pre_tot_res', 'pre_tot_nonres', 'pre_tot_tot',
        'pre_alb_res', 'pre_alb_nonres', 'pre_alb_tot',
        'pre_ext_res', 'pre_ext_nonres', 'pre_ext_tot'
    ]
    df['anno'] = anno
    return df

# Carica tutti gli anni
frames = []
for anno in ANNI:
    df_anno = carica_anno(FILE_ANNUALE, anno)
    frames.append(df_anno)
    print(f"Anno {anno}: {len(df_anno)} righe caricate")

df_raw = pd.concat(frames, ignore_index=True)
print(f"\nTotale righe: {len(df_raw)}")

Anno 2014: 3531 righe caricate
Anno 2015: 3542 righe caricate
Anno 2016: 3560 righe caricate
Anno 2017: 3742 righe caricate
Anno 2018: 3565 righe caricate
Anno 2019: 3524 righe caricate
Anno 2020: 3322 righe caricate
Anno 2021: 3625 righe caricate
Anno 2022: 3740 righe caricate
Anno 2023: 4725 righe caricate
Anno 2024: 5149 righe caricate

Totale righe: 42025


In [ ]:
#Filtro solo Puglia/lecce
# Filtra solo provincia di Lecce
df_lecce = df_raw[df_raw['provincia'].astype(str).str.upper() == 'LECCE'].copy()

# Sostituisce (*) e valori non numerici con NaN
cols_numeriche = [c for c in df_lecce.columns if c.startswith('arr_') or c.startswith('pre_')]
for col in cols_numeriche:
    df_lecce[col] = pd.to_numeric(df_lecce[col], errors='coerce')

print(f"Comuni provincia di Lecce: {df_lecce['comune'].nunique()}")
print(f"Righe totali: {len(df_lecce)}")
print(f"\nComuni trovati:")
# Converti la colonna 'comune' in stringa per gestire eventuali valori float (NaN) prima di ordinare
print(sorted(df_lecce['comune'].astype(str).unique()))

Comuni provincia di Lecce: 94
Righe totali: 766

Comuni trovati:
['Alessano', 'Alezio', 'Alliste', 'Altri comuni della Provincia di LECCE', 'Altri comuni della provincia di LECCE', 'Altri comuni della provincia di Lecce', 'Andrano', 'Aradeo', 'Arnesano', 'Bagnolo del Salento', 'Calimera', 'Campi Salentina', 'Cannole', 'Caprarica di Lecce', 'Carmiano', 'Carpignano Salentino', 'Casarano', "Castrignano de' Greci", 'Castrignano del Capo', 'Castro', 'Cavallino', 'Collepasso', 'Copertino', "Corigliano d'Otranto", 'Corsano', 'Cursi', 'Cutrofiano', 'Diso', 'Gagliano del Capo', 'Galatina', 'Galatone', 'Gallipoli', 'Giuggianello', 'Giurdignano', 'Guagnano', 'Lecce', 'Lequile', 'Leverano', 'Lizzanello', 'Maglie', 'Martano', 'Matino', 'Melendugno', 'Melissano', 'Minervino di Lecce', 'Monteroni di Lecce', 'Montesano Salentino', 'Morciano di Leuca', 'Muro Leccese', 'Nardò', 'Neviano', 'Nociglia', 'Novoli', 'Ortelle', 'Otranto', 'Palmariggi', 'Parabita', 'Patù', 'Poggiardo', 'Porto Cesareo', 'Presicc

In [ ]:
# Rimuove righe con comune NaN
df_lecce = df_lecce[df_lecce['comune'].notna()].copy()

# Uniforma le varianti di "Altri comuni"
df_lecce['comune'] = df_lecce['comune'].str.strip()
df_lecce['comune'] = df_lecce['comune'].replace({
    'Altri comuni della Provincia di LECCE': 'Altri comuni della provincia di LECCE',
    'Altri comuni della provincia di Lecce': 'Altri comuni della provincia di LECCE',
})

# Verifica
print(f"Righe dopo pulizia: {len(df_lecce)}")
print(f"Comuni unici: {df_lecce['comune'].nunique()}")

# Controlla valori mancanti nelle colonne principali
print(f"\nValori mancanti in arr_tot_tot: {df_lecce['arr_tot_tot'].isna().sum()}")
print(f"Valori mancanti in pre_tot_tot: {df_lecce['pre_tot_tot'].isna().sum()}")

Righe dopo pulizia: 760
Comuni unici: 92

Valori mancanti in arr_tot_tot: 0
Valori mancanti in pre_tot_tot: 0


In [ ]:
#Salvare il file pulito
from google.colab import files

OUTPUT = 'lecce_turismo_pulito.csv'
df_lecce.to_csv(OUTPUT, index=False, encoding='utf-8-sig')

# Scarica il file sul tuo PC
files.download(OUTPUT)
print(f"File salvato e scaricato: {OUTPUT}")
print(f"Shape finale: {df_lecce.shape}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File salvato e scaricato: lecce_turismo_pulito.csv
Shape finale: (760, 27)


In [ ]:
# ============================================================
# NOTEBOOK 1 - RACCOLTA E PULIZIA DATI
# ============================================================
# Fonte: Istat - Movimento dei clienti negli esercizi ricettivi
# Dataset: Dati comunali 2014-2024
# Territorio: Provincia di Lecce (Salento)
# Output: lecce_turismo_pulito.csv
#
# Il dataset contiene arrivi e presenze per 92 comuni
# suddivisi per tipologia ricettiva (alberghiero/extra-alberghiero)
# e provenienza (residenti/non residenti)
# Periodo: 2014-2024 (11 anni, dati annuali)
# ============================================================

print("Dataset pronto per l'analisi esplorativa (Notebook 2)")

Dataset pronto per l'analisi esplorativa (Notebook 2)
